In [ ]:
## Allan - EDA ##
### Exploring the Effects of Pollutants on Asthma ER Visits across NYC ###

In [ ]:
import matplotlib.pyplot as plt
import requests
import pandas as pd

In [ ]:
### 📡 Step 1: Data Collection ###
*Pulling data from the NYC Open Data API — fetching all available records using pagination so we go beyond the default 1,000-row limit.*

In [ ]:
base_url = "https://data.cityofnewyork.us/resource/c3uy-2p5r.json"
limit = 1000
offset = 0
all_records = []

while True:
    params = {"$limit": limit, "$offset": offset}
    response = requests.get(base_url, params=params)
    batch = response.json()
    if not batch:
        break
    all_records.extend(batch)
    offset += limit

df = pd.DataFrame(all_records)
print(f"Total records fetched: {df.shape[0]}")
print(f"Columns: {df.columns.tolist()}")

In [ ]:
df.head()

In [ ]:
df['data_value'] = pd.to_numeric(df['data_value'], errors='coerce')

print("Unique pollutants:")
print(df['name'].unique())

In [ ]:
### 🔍 Step 2: Filter for Key Indicators ###
*We narrow the dataset to focus on Fine Particles (PM2.5) — our primary independent variable — and Asthma Emergency Department Visits, our dependent variable.*

In [ ]:
pm25_df = df[
    (df['name'].str.contains('Fine particles', case=False, na=False)) &
    (df['geo_type_name'] == 'UHF42')
].copy()

print(f"PM2.5 records (UHF42): {pm25_df.shape[0]}")
print(pm25_df[['geo_place_name', 'time_period', 'data_value']].head(10))

In [ ]:
asthma_df = df[
    (df['name'].str.contains('Asthma', case=False, na=False)) &
    (df['geo_type_name'] == 'Borough')
].copy()

print(f"Asthma ER visit records (Borough): {asthma_df.shape[0]}")
print(asthma_df[['geo_place_name', 'time_period', 'data_value', 'name']].drop_duplicates('name'))

In [ ]:
### 📊 Step 3: Univariate Analysis — PM2.5 Distribution ###
*Exploring how PM2.5 levels are distributed across neighbourhoods.*

In [ ]:
plt.figure(figsize=(10, 5))
pm25_df['data_value'].dropna().hist(bins=30, color='steelblue', edgecolor='black')
plt.title('Distribution of PM2.5 Concentration Values Across NYC Neighbourhoods')
plt.xlabel('PM2.5 Concentration (μg/m³)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
print("PM2.5 Summary Statistics:")
pm25_df['data_value'].describe()

In [ ]:
### 🗺️ Step 4: Bivariate Analysis — PM2.5 by Neighbourhood ###
*Which NYC neighbourhoods have the highest PM2.5 levels? We rank the top 15 using mean concentration.*

In [ ]:
top_pm25 = (
    pm25_df.groupby('geo_place_name')['data_value']
    .mean()
    .sort_values(ascending=False)
    .head(15)
)

top_pm25.plot(kind='barh', color='steelblue', edgecolor='black', figsize=(10, 7))
plt.title('Top 15 NYC Neighbourhoods by Average PM2.5 Concentration')
plt.xlabel('Average PM2.5 (μg/m³)')
plt.ylabel('Neighbourhood')
plt.tight_layout()
plt.show()

In [ ]:
### 📅 Step 5: Trend Analysis — PM2.5 Over Time ###
*How have PM2.5 levels changed across time periods?*

In [ ]:
pm25_time = (
    pm25_df.groupby('time_period')['data_value']
    .mean()
    .sort_values(ascending=False)
)

pm25_time.plot(kind='bar', color='steelblue', edgecolor='black', figsize=(12, 5))
plt.title('Average PM2.5 Concentration by Time Period')
plt.xlabel('Time Period')
plt.ylabel('Average PM2.5 (μg/m³)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()